## 1. Before You Begin

MAI-Image-2.5-Flash is the production-efficiency variant of the MAI-Image-2.5 family.
It delivers text-to-image generation at lower latency and cost per image, making it the
right choice when throughput and cost matter more than maximum fidelity.

**Use Flash when:**
- Generating large volumes of images (e-commerce, marketing asset pipelines)
- Latency per image is more important than portrait or text-rendering precision
- Running concurrent batch jobs under a cost budget

**Prerequisites**

1. A Microsoft Foundry project with MAI-Image-2.5-Flash deployed.
   See [models/quickstart/](../../quickstart/README.md) for first-time setup.
2. The three environment variables listed in section 2.
3. `requests` installed: `pip install requests`

**API constraints** ([source: Learn docs](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python))

| Rule | Value |
|---|---|
| Minimum dimension (each) | 768 px |
| Maximum total pixels | 1,048,576 |
| Output format | PNG only |
| Auth header | `api-key` |


## 2. Set up your environment

The cell below verifies the three required variables and derives `BASE_ENDPOINT`.
It also imports `time` and `concurrent.futures`, used in later sections for
throughput measurement and concurrent requests.


In [ ]:
%pip install requests python-dotenv --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv()

# Your .env file needs the environment variables below
# MICROSOFT_FOUNDRY_ENDPOINT= <https://>your-project>.services.ai.azure.com>
# MICROSOFT_FOUNDRY_API_KEY=XXXX
# AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT=MAI-Image-2.5-Flash

In [ ]:
import os, base64, time
import requests
import concurrent.futures
from pathlib import Path
from urllib.parse import urlparse
from IPython.display import Image as IPyImage, display

REQUIRED = {
    "MICROSOFT_FOUNDRY_ENDPOINT":             "Foundry project endpoint",
    "MICROSOFT_FOUNDRY_API_KEY":               "AIServices API key",
    "AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT":     "MAI-Image-2.5-Flash deployment name",
}
missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise EnvironmentError(f"Set these env vars before continuing: {missing}")

parsed        = urlparse(os.environ["MICROSOFT_FOUNDRY_ENDPOINT"])
BASE_ENDPOINT = f"{parsed.scheme}://{parsed.netloc}"
API_KEY       = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
DEPLOYMENT    = os.environ["AZURE_MAI_IMAGE_25_FLASH_DEPLOYMENT"]

print(f"Base endpoint : {BASE_ENDPOINT}")
print(f"Deployment    : {DEPLOYMENT}")
print("Environment check passed.")


## 3. Configure the client

Flash supports both `/mai/v1/images/generations` and `/mai/v1/images/edits`. The shared helpers preserve service error details, avoid replacing output files, decode returned PNG images, and display results inline.

In [ ]:
GEN_URL = f"{BASE_ENDPOINT}/mai/v1/images/generations"

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)


def generate_image(prompt: str, width: int = 1024, height: int = 1024) -> str:
    """POST to /mai/v1/images/generations; return base64 PNG string."""
    assert width >= 768 and height >= 768, "Each dimension must be ≥ 768 px"
    assert width * height <= 1_048_576,    "width × height must be ≤ 1,048,576"
    resp = requests.post(
        GEN_URL,
        headers={"Content-Type": "application/json", "api-key": API_KEY},
        json={"model": DEPLOYMENT, "prompt": prompt, "width": width, "height": height},
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["data"][0]["b64_json"]


def show(b64: str, save_as: str | None = None) -> None:
    """Decode base64 PNG, optionally save, and display inline."""
    raw = base64.b64decode(b64)
    if save_as:
        out = OUT_DIR / save_as
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_bytes(raw)
        print(f"Saved → {out}")
    display(IPyImage(data=raw))

import base64
import binascii
import time
from pathlib import Path

import requests

cell_started = time.perf_counter()
print(f'Model: {DEPLOYMENT}')

REQUEST_TIMEOUT = 300

def versioned_path(path: str) -> str:
    candidate = Path(path)
    if not candidate.exists():
        return str(candidate)
    for version in range(2, 10_000):
        next_path = candidate.with_stem(f'{candidate.stem}_v{version}')
        if not next_path.exists():
            return str(next_path)
    raise RuntimeError('Could not find an available output path.')

def post_image_api(route: str, *, json=None, data=None, files=None) -> dict:
    started = time.perf_counter()
    print(f'Model: {DEPLOYMENT}')
    response = requests.post(f'{BASE_ENDPOINT}/mai/v1/images/{route}', headers={'api-key': API_KEY}, json=json, data=data, files=files, timeout=REQUEST_TIMEOUT)
    if not response.ok:
        raise requests.HTTPError(f'{response.status_code} {response.reason}: {response.text}', response=response)
    print(f'API runtime: {time.perf_counter() - started:.2f}s')
    return response.json()

def save_images(result: dict, output_path: str) -> list[str]:
    images = [item['b64_json'] for item in result.get('data', []) if item.get('b64_json')]
    if not images:
        raise ValueError(f'Unexpected response format: {result}')
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    for index, encoded in enumerate(images, start=1):
        path = output if len(images) == 1 else output.with_stem(f'{output.stem}_{index}')
        try:
            path.write_bytes(base64.b64decode(encoded))
        except (binascii.Error, ValueError) as exc:
            raise ValueError(f'Could not decode image {index}.') from exc
        paths.append(str(path.resolve()))
        print(f'Saved: {paths[-1]}')
    return paths

print(f'Total runtime: {time.perf_counter() - cell_started:.2f}s')

## 4. Generate a batch of images

A batch is a list of prompts processed sequentially.
This establishes a baseline before parallelising in section 6.

The prompts below represent a product photography pipeline:
eight product variants, one image each.


In [ ]:
PRODUCT_PROMPTS = [
    "Studio shot of a slate-grey water bottle on white background, soft-box lighting",
    "Studio shot of a forest-green water bottle on white background, soft-box lighting",
    "Studio shot of a coral-red water bottle on white background, soft-box lighting",
    "Studio shot of a midnight-blue water bottle on white background, soft-box lighting",
    "Studio shot of a sand-beige water bottle on white background, soft-box lighting",
    "Studio shot of a charcoal-black water bottle on white background, soft-box lighting",
    "Studio shot of a cloud-white water bottle on white background, soft-box lighting",
    "Studio shot of a burnt-orange water bottle on white background, soft-box lighting",
]

results_sequential = []
t_start = time.monotonic()

for i, prompt in enumerate(PRODUCT_PROMPTS):
    b64 = generate_image(prompt, width=1024, height=1024)
    results_sequential.append(b64)
    show(b64, save_as=f"sequential_batch/batch-seq-{i:02d}.png")

elapsed = time.monotonic() - t_start
print(f"\nSequential batch: {len(PRODUCT_PROMPTS)} images in {elapsed:.1f}s "
      f"({elapsed/len(PRODUCT_PROMPTS):.1f}s / image)")


## 5. Measure throughput and estimate cost

Before parallelising, record the single-request latency distribution.
This gives a baseline to compare against the concurrent run in section 6.

The cell below generates one image per prompt and records wall-clock time per call.
At current Flash pricing, estimate cost as `n_images × price_per_image`.


In [ ]:
PRICE_PER_IMAGE = 0.04   # update if pricing changes — check model page for current rate

latencies = []
sample_prompts = PRODUCT_PROMPTS[:4]   # 4 samples for a faster measurement

for prompt in sample_prompts:
    t0 = time.monotonic()
    b64 = generate_image(prompt, width=1024, height=1024)
    latencies.append(time.monotonic() - t0)

avg_latency = sum(latencies) / len(latencies)
print(f"Samples        : {len(latencies)}")
print(f"Avg latency    : {avg_latency:.2f}s / image")
print(f"Min / Max      : {min(latencies):.2f}s / {max(latencies):.2f}s")
print()

for n in (10, 100, 1_000, 10_000):
    est_cost  = n * PRICE_PER_IMAGE
    est_time  = n * avg_latency
    print(f"  {n:>6} images → ${est_cost:>8.2f}  |  sequential ETA ≈ {est_time/60:,.1f} min")


## 6. Run concurrent requests

`concurrent.futures.ThreadPoolExecutor` issues requests in parallel.
Each worker is a blocking HTTP call, so threads (not processes) work here.

**Choose `max_workers` carefully:**
- Start with 4–8 workers to avoid rate-limit errors
- Check your Foundry quota page for the TPM / RPM limits on your deployment
- Increase gradually and re-run this cell to find your practical ceiling


In [ ]:
MAX_WORKERS = 4   # increase after confirming your quota allows it

def _generate(args):
    idx, prompt = args
    t0 = time.monotonic()
    b64 = generate_image(prompt, width=1024, height=1024)
    elapsed = time.monotonic() - t0
    return idx, b64, elapsed

t_start = time.monotonic()

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = list(executor.map(_generate, enumerate(PRODUCT_PROMPTS)))

total_elapsed = time.monotonic() - t_start

for idx, b64, per_latency in futures:
    show(b64, save_as=f"concurrent_batch/batch-par-{idx:02d}.png")

images_per_sec = len(PRODUCT_PROMPTS) / total_elapsed
print(f"\nConcurrent batch ({MAX_WORKERS} workers): "
      f"{len(PRODUCT_PROMPTS)} images in {total_elapsed:.1f}s "
      f"({images_per_sec:.2f} images/s)")


## 7. Create multiple background options whilst maintaining original feature

Keeping the clothing item photorealistic preserves accurate color, texture, condition, and construction details. Editing only the background lets a seller reuse one truthful product image across multiple visual settings without the time and cost of repeated photo shoots.

Use the edits endpoint to place one clothing photo in six marketplace-ready settings while preserving the item. Add a JPEG or PNG at `images/blue-dress.jpg`, or update `CLOTHING_IMAGE` in the next cell. Results are versioned under `output/`.


In [ ]:
cell_started = time.perf_counter()
print(f'Model: {DEPLOYMENT}')

import mimetypes

RESALE_LISTING_BACKGROUNDS = [
    ('paris_apartment', 'a bright Parisian apartment with pale walls, subtle molding, and warm oak flooring'),
    ('fashion_boutique', 'an elegant independent fashion boutique with neutral decor and soft window light'),
    ('minimal_studio', 'a clean warm-white photography studio with a seamless backdrop and soft diffused lighting'),
    ('luxury_wardrobe', 'a refined walk-in wardrobe with muted finishes and an uncluttered editorial look'),
    ('hotel_suite', 'a sophisticated boutique hotel suite with tasteful neutral furnishings and natural daylight'),
    ('stone_townhouse', 'a chic European townhouse interior with pale stone details and understated styling'),
]

def create_resale_listing_background_variations(image_path: str, output_dir: str = 'output') -> list[str]:
    source = Path(image_path)
    if not source.is_file():
        raise FileNotFoundError(f'Clothing image not found: {source}')
    mime_type = mimetypes.guess_type(source.name)[0]
    if mime_type not in {'image/jpeg', 'image/png'}:
        raise ValueError('The clothing image must be a JPEG or PNG image.')

    def create_variation(item: tuple[int, tuple[str, str]]) -> list[str]:
        index, (name, background) = item
        prompt = ('Edit only the background of this secondhand clothing photo. Place the clothing item in ' + background + '. ' 'Keep the item and every foreground detail unchanged: shape, cut, proportions, color, pattern, fabric texture, seams, labels, embellishments, condition, and fit. ' 'Do not redesign, retouch, reshape, recolor, or add anything to the item. Match the original perspective and use physically realistic lighting and shadows. ' 'Create an authentic professional product photograph suitable for an online clothing resale listing. No text, logos, borders, or watermarks.')
        output_path = versioned_path(str(Path(output_dir) / f'{source.stem}_resale_listing_backgrounds' / f'{index:02d}_{name}.png'))
        print(f'Creating variation {index}/{len(RESALE_LISTING_BACKGROUNDS)}: {name}')
        with source.open('rb') as image_file:
            result = post_image_api('edits', data={'model': DEPLOYMENT, 'prompt': prompt}, files={'image': (source.name, image_file, mime_type)})
        return save_images(result, output_path)

    indexed_backgrounds = list(enumerate(RESALE_LISTING_BACKGROUNDS, start=1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        variation_paths = executor.map(create_variation, indexed_backgrounds)
    return [path for paths in variation_paths for path in paths]

CLOTHING_IMAGE = 'images/blue-dress.jpg'  # Replace with your clothing image.
resale_listing_paths = create_resale_listing_background_variations(CLOTHING_IMAGE)
for path in resale_listing_paths:
    display(IPyImage(filename=path))
print(f'Total runtime: {time.perf_counter() - cell_started:.2f}s')

## 8. Generate visual mood boards

Generate six art directions for **The AI Engineer's desk** while keeping the composition rules fixed. This makes the visual treatment easier to compare across materials, lighting, and accent colors.

In [ ]:
cell_started = time.perf_counter()
print(f'Model: {DEPLOYMENT}')

AI_ENGINEER_DESK_DIRECTIONS = [
    ('precision_lab', 'precision research lab, white ceramic, brushed aluminum, cool daylight, cyan accents'),
    ('warm_startup', 'warm creative startup studio, walnut desk, soft amber light, plants, coral and teal accents'),
    ('midnight_ops', 'midnight AI operations workspace, charcoal surfaces, focused monitor glow, electric blue accents'),
    ('calm_minimal', 'calm minimalist workspace, pale ash wood, matte white tools, diffused morning light, sage accents'),
    ('analog_digital', 'analog-meets-digital workshop, notebooks, technical sketches, mechanical keyboard, moss and rust tones'),
    ('future_luxury', 'quiet futuristic luxury office, smoked glass, black metal, sculptural lighting, silver accents'),
]

def generate_ai_engineer_desk_mood_boards(output_dir: str = 'output/ai_engineers_desk_mood_boards') -> list[str]:
    def generate_mood_board(item: tuple[int, tuple[str, str]]) -> list[str]:
        index, (name, direction) = item
        prompt = (
            "Create a sophisticated visual mood board for the concept 'The AI Engineer's desk'. "
            f'Art direction: {direction}. Compose a cohesive editorial collage with one hero workspace image and smaller supporting frames showing tactile materials, desk objects, computing hardware, abstract neural-network diagrams, lighting details, and a restrained row of color swatches. '
            'Professional design presentation, photorealistic product photography, precise grid, generous spacing, coherent lighting, premium art-direction quality, square composition. No people, brand logos, watermarks, captions, labels, or legible text.'
        )
        print(f'Creating mood board {index}/{len(AI_ENGINEER_DESK_DIRECTIONS)}: {name}')
        result = post_image_api('generations', json={'model': DEPLOYMENT, 'prompt': prompt, 'width': 1024, 'height': 1024})
        output_path = versioned_path(str(Path(output_dir) / f'{index:02d}_{name}.png'))
        return save_images(result, output_path)

    indexed_directions = list(enumerate(AI_ENGINEER_DESK_DIRECTIONS, start=1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        mood_board_paths = executor.map(generate_mood_board, indexed_directions)
    return [path for paths in mood_board_paths for path in paths]

mood_board_paths = generate_ai_engineer_desk_mood_boards()
for path in mood_board_paths:
    display(IPyImage(filename=path))
print(f'Total runtime: {time.perf_counter() - cell_started:.2f}s')

## 9. Your Turn to Explore

1. Double `MAX_WORKERS` and compare throughput, latency, and rate-limit responses.
2. Test resale backgrounds with different lighting and record which item details remain stable.
3. Combine two mood-board compositions with three art directions and compare consistency.

In [ ]:
# Your experiments here


## 10. Summary

You used MAI-Image-2.5-Flash to:

- Process product-photography prompts sequentially and concurrently
- Measure latency, estimate cost, and compare throughput
- Create six options of backgrounds for a source image
- Generate six coordinated visual mood boards

Choose Flash for image pipelines where latency, throughput, and cost matter more than maximum fidelity.

**Next steps**

- [MAI-Image-2.5 capsule](../mai-image-2.5/) - baseline model with image editing
- [MAI-Image-2.5-Pro capsule](../mai-image-2.5-pro/) - portrait quality and text rendering
- [Image generation primer](../../../docs/primers/image-generation.md)

## 11. References

- [Deploy and use MAI image models in Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-mai-image?tabs=python) - official generations and edits API guidance
- [Build 2026 MAI keynote transcript](https://microsoft.ai/news/microsoft-build-2026-mai-keynote-transcript/) - introduction to the MAI model family
- [MAI-Image-2.5 model page](https://microsoft.ai/models/mai-image-2-5/) - model overview and capability comparison